# Construire un plugin TethysDash — le graphique d'exemple

Tout ce qui s'affiche sur un tableau de bord TethysDash est dessiné par un **plugin** :
un paquet Python installable que l'application découvre toute seule. Ce notebook
construit le plus simple qui soit utile, en partant de zéro — un graphique en courbes —
puis lui ajoute un argument et une barre de progression.

La question posée est la suivante :

> **Quelle est la plus petite quantité de Python qui place un nouveau graphique dans
> le sélecteur de visualisations ?**

La réponse est une classe avec quatre attributs et une méthode. Tout le reste est
facultatif.

**Ce qu'il faut en retenir**

- ce que `run()` est réellement tenue de retourner, et comment le vérifier soi-même
- comment la déclaration d'`args` procure des contrôles d'interface sans écrire de
  formulaire
- pourquoi `get_arg()` est plus sûr que de lire l'argument sur `self`
- ce que signifie enregistrer un plugin, et pourquoi cela ne demande aucune
  modification de TethysDash

**Déroulement**

1. Récupérer les données et construire le graphique avec plotly, comme partout ailleurs
2. Regarder ce que produit `to_json()` — c'est *cela* la valeur de retour du plugin
3. L'envelopper dans une classe de plugin et l'exécuter ici, sans serveur
4. Ajouter un argument, et voir d'où vient le contrôle d'interface
5. Ajouter des mises à jour de progression
6. L'enregistrer pour que l'application puisse le trouver

Un vrai plugin importe sa classe de base depuis TethysDash :

```python
from tethysapp.tethysdash.plugin_helpers import TethysDashPlugin
```

Ce notebook définit plutôt une **doublure** respectant le même contrat, afin que chaque
cellule s'exécute sur un portable où seul plotly est installé. Cette unique ligne
d'import est la seule différence entre ce qui suit et un vrai fichier de plugin.

In [1]:
!pip install plotly "nbformat>=4.2.0"

In [2]:
import json

import plotly.express as px
import plotly.graph_objects as go

VALID_TYPES = ["plotly", "table", "image", "card", "text", "variable_input",
               "map", "map_layer", "custom", "imageCollection"]

class TethysDashPlugin:
    """Teaching stand-in for the real base class, with the same contract.

    Mirrors what the real one does at construction: it validates the four
    required attributes, rejects an unknown `type`, and exposes the supplied
    arguments through both `get_arg()` and attribute access.

    `send_update()` here just prints. The real one publishes over a WebSocket and
    needs the request context the app attaches when it invokes a plugin, so it
    cannot be called outside a running server -- which is exactly why this
    notebook uses a stand-in rather than importing the real class.
    """

    args = {}

    def __init__(self, **kwargs):
        for attr in ("name", "type", "label", "group"):
            if getattr(self, attr, None) in (None, ""):
                raise ValueError(f"Plugin must have a {attr} attribute defined.")
        if self.type not in VALID_TYPES:
            raise ValueError(
                f"Plugin type '{self.type}' is not valid. "
                f"Must be one of: {', '.join(VALID_TYPES)}"
            )
        self.received_args = dict(kwargs)
        for key, value in kwargs.items():
            setattr(self, key, value)

    def get_arg(self, name, default=None):
        return self.received_args.get(name, default)

    def send_update(self, message, percentage_complete=None, layer_id=None):
        pct = "" if percentage_complete is None else f" [{percentage_complete}%]"
        print(f"  progress{pct}: {message}")

print(f"stand-in ready | {len(VALID_TYPES)} valid plugin types")

stand-in ready | 10 valid plugin types


## 1. Le graphique, construit de la manière habituelle

Rien dans cette étape n'est propre à TethysDash. C'est le tracé que vous écririez dans
n'importe quel notebook — et c'est bien là l'idée : un plugin, c'est le code que vous
avez déjà, enveloppé pour que l'application puisse l'appeler.

In [4]:
df = px.data.gapminder()
print(f"{len(df):,} rows, {df.country.nunique()} countries, "
      f"{df.year.min()}-{df.year.max()}")
df[df.country == "Haiti"].head(3)

1,704 rows, 142 countries, 1952-2007


,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
636,Haiti,Americas,1952,37.579,3201488,1840.366939,HTI,332
637,Haiti,Americas,1957,40.696,3507701,1726.887882,HTI,332
638,Haiti,Americas,1962,43.590,3880130,1796.589032,HTI,332


In [5]:
asia = df.query("continent == 'Americas'")
fig = px.line(asia, x="year", y="lifeExp", color="country", symbol="country")
fig

## 2. Ce que `run()` doit retourner

Un plugin déclare `type = "plotly"`, et ce choix détermine la forme de sa valeur de
retour. Pour `plotly`, le contrat est la figure sous forme de **dictionnaire simple** —
exactement ce que produit `to_json()`.

Regardez les clés plutôt que le contenu :

In [6]:
payload = json.loads(fig.to_json())

print("top-level keys:", list(payload))
print(f"  data:   {len(payload['data'])} traces")
print(f"  layout: {len(payload['layout'])} keys -> {list(payload['layout'])[:6]}...")
print(f"\nserialised size: {len(fig.to_json()) / 1024:.0f} KB")

top-level keys: ['data', 'layout']
  data:   25 traces
  layout: 5 keys -> ['template', 'xaxis', 'yaxis', 'legend', 'margin']...

serialised size: 21 KB


Ce dictionnaire constitue toute l'interface. Le plugin ne dessine rien et ne touche
pas au navigateur — il retourne des données, et c'est le frontend qui effectue le rendu.

Ce qui signifie que le contrat est vérifiable sans serveur. Faites l'aller-retour du
contenu vers une figure : s'il se dessine, un vrai tableau de bord dessinerait la même
chose.

In [7]:
go.Figure(payload)

## 3. Le plugin minimal

Quatre attributs obligatoires et une méthode :

| attribut | à quoi il sert |
|---|---|
| `name` | le nom d'installation et de driver — doit correspondre au point d'entrée |
| `group` | regroupe le plugin dans le sélecteur de visualisations |
| `label` | le nom affiché dans l'application |
| `type` | choisit le moteur de rendu, et dicte donc ce que `run()` retourne |

En omettre un seul lève une erreur à la construction : un plugin défectueux échoue donc
bruyamment plutôt que d'apparaître à moitié dans l'application.

In [8]:
class PlotExample(TethysDashPlugin):
    name = "plot_example"
    group = "Example"
    label = "Example Plot"
    type = "plotly"

    def run(self):
        frame = px.data.gapminder().query("continent == 'Americas'")
        figure = px.line(frame, x="year", y="lifeExp",
                         color="country", symbol="country")
        return json.loads(figure.to_json())


plugin = PlotExample()
result = plugin.run()
print(f"run() returned {type(result).__name__} with keys {list(result)}")
print(f"  {len(result['data'])} traces")

run() returned dict with keys ['data', 'layout']
  25 traces


Instancier la classe et appeler `run()`, c'est aussi ainsi qu'on teste un plugin — sans
serveur, sans tableau de bord, sans navigateur. Si `run()` retourne la bonne forme, la
visualisation fonctionne.

In [9]:
# What happens when a required attribute is missing.
class Broken(TethysDashPlugin):
    name = "broken"
    type = "plotly"
    label = "Broken"
    # group is missing

try:
    Broken()
except ValueError as err:
    print(f"ValueError: {err}")

ValueError: Plugin must have a group attribute defined.


## 4. Ajouter un argument

C'est la déclaration d'`args` qui produit les contrôles d'interface. L'auteur du tableau
de bord ne voit jamais un formulaire que vous auriez écrit — TethysDash génère le champ
de saisie à partir du type déclaré, et l'étiquette à partir du nom de l'argument
lui-même.

`{"continent": "text"}` devient une zone de texte étiquetée **Continent**.

In [14]:
class PlotByContinent(TethysDashPlugin):
    name = "plot_by_continent"
    group = "Example"
    label = "Plot by Continent"
    type = "plotly"
    args = {"country": "text"}      # -> one text input, labelled "Country"

    def run(self):
        country = self.get_arg("country", "Haiti")
        frame = px.data.gapminder().query(f"country == '{country}'")
        figure = px.line(frame, x="year", y="lifeExp",
                         color="country", symbol="country")
        figure.update_layout(title=f"Life expectancy — {country}")
        return json.loads(figure.to_json())


# The app passes the configured values in; here we pass them by hand.
for country in ("Haiti", "Guatemala", "Jamaica"):
    out = PlotByContinent(country=country).run()
    print(f"{country:<8} {len(out['data']):>2} traces  "
          f"title={out['layout']['title']['text']!r}")

Haiti     1 traces  title='Life expectancy — Haiti'
Guatemala  1 traces  title='Life expectancy — Guatemala'
Jamaica   1 traces  title='Life expectancy — Jamaica'


In [16]:
go.Figure(PlotByContinent(country="Haiti").run())

### Lisez les arguments avec `get_arg()`, pas sur `self`

L'exemple des diapositives utilise `self.continent`, et pour un nom simple cela
fonctionne — le framework définit chaque argument fourni comme un attribut. Mais
**`get_arg()` est à privilégier**, pour deux raisons :

- **Les arguments imbriqués ont des noms pointés**, comme `transect_location.location`.
  Python ne sait pas résoudre un attribut pointé, donc `self.transect_location.location`
  échoue — silencieusement, dans le pire des cas.
- **`get_arg()` accepte une valeur par défaut.** Un argument laissé vide par l'auteur est
  tout simplement absent : l'accès par attribut lève alors `AttributeError`, là où
  `get_arg("continent", "Asia")` poursuit son chemin.

`args` ne peut pas non plus reprendre les noms des propriétés du plugin lui-même —
`type`, `label`, `group`, `tags` et les autres sont réservés, et entrer en collision avec
l'un d'eux lève une erreur à la construction.

## 5. Les mises à jour de progression

Un plugin qui prend quelques secondes devrait le faire savoir. `send_update()` diffuse un
message — et éventuellement un pourcentage — vers l'élément du tableau de bord par
WebSocket, ce qui transforme une tuile vide en barre de progression.

Ici, la doublure se contente d'afficher, pour que vous puissiez voir la séquence.

Une remarque pratique : le vrai `send_update()` a besoin du contexte de requête que
l'application attache lorsqu'elle appelle un plugin, il ne fonctionne donc qu'à
l'intérieur d'un serveur en cours d'exécution. L'appeler depuis un script lève
`AttributeError`. Ce n'est pas un problème en pratique — cela signifie simplement que le
rapport de progression est la seule partie d'un plugin que vous ne pouvez pas exercer en
dehors de l'application.

In [18]:
class PlotWithProgress(TethysDashPlugin):
    name = "plot_with_progress"
    group = "Example"
    label = "Plot with Progress"
    type = "plotly"
    args = {"continent": "text"}

    def run(self):
        continent = self.get_arg("continent", "Americas")

        self.send_update("Gathering data from plotly", percentage_complete=25)
        frame = px.data.gapminder().query(f"continent == '{continent}'")

        self.send_update("Plotting data", percentage_complete=75)
        figure = px.line(frame, x="year", y="lifeExp",
                         color="country", symbol="country")

        self.send_update("Done", percentage_complete=100)
        return json.loads(figure.to_json())


print("running PlotWithProgress(continent='Africa')")
out = PlotWithProgress(continent="Africa").run()
print(f"-> {len(out['data'])} traces")

running PlotWithProgress(continent='Africa')
  progress [25%]: Gathering data from plotly
  progress [75%]: Plotting data
  progress [100%]: Done
-> 52 traces


Pour les plugins d'inondation des exercices 2 et 3, ce n'est pas cosmétique.
Échantillonner la profondeur sur 5 000 bâtiments prend quelques secondes, et sans
progression la tuile a l'air en panne plutôt qu'occupée.

## 6. L'enregistrer

Le plugin existe, mais l'application ne le voit pas encore. L'enregistrement tient en un
seul point d'entrée dans le `pyproject.toml` du paquet — TethysDash n'est jamais modifié.

```toml
[project.entry-points."intake.drivers"]
plot_example = "my_plugin.source:PlotExample"
```

Puis :

```bash
pip install .        # into the environment TethysDash runs in
```

Redémarrez l'application et le plugin apparaît dans la liste déroulante **Visualization
Type**, sous le `group` qu'il a déclaré. Six choses se produisent, dont aucune à
l'intérieur de TethysDash :

1. Le point d'entrée est déclaré dans `pyproject.toml` (ou `setup.py`)
2. `pip install` place le paquet dans l'environnement de TethysDash
3. Intake l'enregistre automatiquement — `open_plot_example` devient appelable
4. Le plugin apparaît dans le sélecteur de visualisations
5. Un dossier `static/` de vignettes le rend reconnaissable parmi beaucoup d'autres
6. Le tester revient à instancier la classe et appeler `run()`, comme ci-dessus

La conséquence institutionnelle mérite d'être énoncée clairement : **installer un plugin,
c'est installer un paquet Python.** Cela ne demande aucune nouvelle procédure, aucune
modification de TethysDash, et rien qui empêche de mettre la plateforme à niveau plus
tard.

### Les propriétés facultatives

| propriété | défaut | à quoi elle sert |
|---|---|---|
| `args` | `{}` | schéma des arguments → champs de saisie générés |
| `tags` | `[]` | recherche et découverte |
| `description` | `""` | affichée aux utilisateurs à côté de la sélection |
| `restricted` | `False` | réserve le plugin aux utilisateurs autorisés |
| `loading_icon` | `True` | indicateur d'attente pendant l'exécution du plugin |
| `attribution` | `""` | mention de la source des données, dessinée sur le widget |
| `dynamic_map_layer` | `False` | active `fetch_features()` pour les couches cartographiques dynamiques |

Deux comptent dans un cadre institutionnel : `restricted`, qui place un plugin derrière
des permissions, et `attribution`, qui affiche automatiquement la mention de la source
des données — souvent une exigence formelle lorsqu'on publie les données d'autrui.

### Choses à essayer

- Remplacez `type` par une valeur invalide et observez l'erreur. Essayez ensuite
  `"table"` et faites retourner à `run()` un `{"title": ..., "data": [ ... ]}` à la
  place — la même classe, un autre moteur de rendu.
- Déclarez `args = {"continent": "text", "year": "number"}` et filtrez sur les deux.
  Qu'afficherait l'interface ?
- Essayez `args = {"type": "text"}` et lisez l'erreur. Pourquoi ce nom est-il réservé ?
- Supprimez la valeur par défaut de `get_arg` et instanciez sans `continent`. Comparez
  l'échec avec ce qu'aurait donné `self.continent`.
- Le graphique est redessiné intégralement à chaque appel. Quelles parties de `run()`
  mettriez-vous en cache si les données venaient d'une source lente plutôt que de plotly ?